# SQLCLEAN — Local SQL-backed Data Selection Pipeline

Reproducible, end-to-end **selection** for the wav2vec2 / WavLM speech-emotion pipeline, backed by a
**local SQLite file** instead of the cloud MySQL used by `sqltransfer.ipynb` / `wav2vec2_processing.ipynb`.
No credentials, no server, no network — just `Notebooks/ser_local.db`.

All of the "which clips make the cut" logic lives **here**, once:

1. Fixed random seeds (numpy / random / torch / split = **42**)
2. Build a **metadata table** (one row per file) with `duplicate_flagged` / `corrupted_flagged` / source / duration
3. **Filter to the clean working set**: CREMA-D + RAVDESS only, drop the misspelled 'Suprised', exclude corrupted/duplicate files, keep 1–5 s clips
4. **Speaker-independent 3-way split** (train / val / test) via `GroupShuffleSplit(test_size=0.15, random_state=42)`, applied twice — no speaker appears in more than one split
5. **Resample** every clean clip to 16 kHz mono, capped at 5 s → `Emotions_w2v2_16k/` (originals untouched)
6. **Push the cleaned + split dataset to local SQLite** (`audio_dataset_w2v2`) — the single source of truth `CleanModel.ipynb` queries
7. Print shapes + class/speaker distributions to **verify reproducibility** across the team

> `CleanModel.ipynb` does **no filtering, no splitting, no dataset download** — it just runs
> `SELECT ... FROM audio_dataset_w2v2 WHERE split = 'train'` and starts training. All "data selection"
> decisions are made once, here, and are reproducible from this file + the raw `../Emotions` folder alone.


In [ ]:
# === Imports + fixed random seeds (run first, every session) ===
import os, re, hashlib, random
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    _t = "/torch"
except ImportError:
    _t = ""
print(f"seeds set to {SEED}  (numpy/random{_t})")


In [ ]:
# === Config ===
EMO_DIR       = Path("../Emotions")              # raw dataset root: <emotion>/*.wav
PROCESSED_DIR = Path("../Emotions_w2v2_16k")     # 16 kHz output (NEVER write into EMO_DIR)
DB_PATH       = Path("ser_local.db")             # local SQLite file, lives next to this notebook
SR            = 16000                             # wav2vec2 / WavLM sampling rate
DUR_MIN, DUR_MAX = 1.0, 5.0                       # keep clips in this duration window (sec)
MAX_SEC       = 5.0                               # cap clip length (no padding; collator pads per batch)

# Cleaned-dataset scope: reduce to these sources, drop these (misspelled / out-of-scope) emotions.
KEEP_SOURCES = ("CREMA-D", "RAVDESS")
DROP_LABELS  = ("Suprised",)

def source_of(name):
    if re.match(r"^\d+-\d+-", name):             return "RAVDESS"
    if re.match(r"^\d{4}_", name):               return "CREMA-D"
    if re.match(r"^(OAF|YAF|OA)_", name):        return "TESS"
    if re.match(r"^[a-z]{1,2}\d+\.wav$", name):  return "SAVEE"
    return "other"

def speaker_of(filename, source):
    # speaker id for GroupShuffleSplit -- same logic CleanModel expects downstream
    if source == "CREMA-D":   return f"CREMA-{filename[:4]}"
    if source == "RAVDESS":   return f"RAVDESS-{filename.split('.')[0].split('-')[-1]}"
    return f"{source}-{filename}"


In [ ]:
# === Database connection (local SQLite file -- no server, no credentials, no network) ===
# A single ser_local.db file next to this notebook. CleanModel.ipynb opens the SAME file (read-only,
# via the same relative path) and expects the `audio_dataset_w2v2` table to already exist.
from sqlalchemy import create_engine, text

engine = create_engine(f"sqlite:///{DB_PATH}")

def recreate_table(name, ddl, index_ddls=()):
    """DROP + CREATE a table, then create indexes (SQLite has no inline INDEX in CREATE TABLE)."""
    with engine.begin() as conn:
        conn.execute(text(f"DROP TABLE IF EXISTS {name}"))
        conn.execute(text(ddl))
        for idx_ddl in index_ddls:
            conn.execute(text(idx_ddl))

with engine.connect() as conn:
    v = conn.execute(text("SELECT sqlite_version()")).scalar()
print(f"OK  local SQLite {v}  ->  {DB_PATH.resolve()}")


In [ ]:
# === Build the metadata table (one row per file) ===
# Reads only file headers (soundfile.info) -> fast. Flags corrupted (unreadable / 0 frames) and
# duplicate (identical bytes) files. Nothing is deleted or moved -- flags only.
def file_md5(p, chunk=1 << 20):
    h = hashlib.md5()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(chunk), b""):
            h.update(blk)
    return h.hexdigest()

rows = []
for emo_path in sorted(p for p in EMO_DIR.iterdir() if p.is_dir()):
    for wav in sorted(emo_path.glob("*.wav")):
        rec = {"filename": wav.name, "file_path": str(wav), "emotion": emo_path.name,
               "source": source_of(wav.name), "file_size": wav.stat().st_size}
        try:
            info = sf.info(str(wav))
            rec["duration_sec"] = round(info.frames / info.samplerate, 3)
            rec["sample_rate"]  = int(info.samplerate)
            rec["channels"]     = int(info.channels)
            rec["corrupted_flagged"] = bool(info.frames == 0)
        except Exception:
            rec.update(duration_sec=None, sample_rate=None, channels=None, corrupted_flagged=True)
        rec["md5"] = file_md5(wav) if not rec["corrupted_flagged"] else None
        rows.append(rec)

meta = pd.DataFrame(rows)
meta["duplicate_flagged"] = meta["md5"].duplicated(keep="first") & meta["md5"].notna()
meta = meta.drop(columns=["md5"])               # hash only needed to compute duplicate_flagged
meta.insert(0, "clip_id", range(1, len(meta) + 1))
print(f"{len(meta)} files | corrupted: {int(meta['corrupted_flagged'].sum())} | "
      f"duplicates: {int(meta['duplicate_flagged'].sum())}")
meta.head()


In [ ]:
# === Filter to the CLEAN working set (in memory) ===
# Clean = not corrupted, not duplicate, CREMA-D + RAVDESS only, 'Suprised' dropped, duration 1-5 s.
clean = meta[(~meta["corrupted_flagged"])
             & (~meta["duplicate_flagged"])
             & (meta["source"].isin(KEEP_SOURCES))
             & (~meta["emotion"].isin(DROP_LABELS))
             & (meta["duration_sec"] >= DUR_MIN)
             & (meta["duration_sec"] <= DUR_MAX)].reset_index(drop=True)
print(f"clean clips: {len(clean)} / {len(meta)} total  "
      f"({', '.join(KEEP_SOURCES)}, no {'/'.join(DROP_LABELS)}, {DUR_MIN}-{DUR_MAX}s)")
print(clean["emotion"].value_counts())


In [ ]:
# === Speaker-INDEPENDENT 3-way split (train / val / test, all speaker-disjoint) ===
# GroupShuffleSplit(test_size=0.15, random_state=42), applied twice -- hold out test, then carve
# val out of the train pool. This is the ONLY place the split happens for this pipeline.
from sklearn.model_selection import GroupShuffleSplit

clean = clean.copy()
clean["speaker"] = [speaker_of(n, s) for n, s in zip(clean["filename"], clean["source"])]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trv_idx, te_idx = next(gss.split(clean, clean["emotion"], groups=clean["speaker"]))
trv  = clean.iloc[trv_idx]
test = clean.iloc[te_idx]

gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_rel, va_rel = next(gss_val.split(trv, trv["emotion"], groups=trv["speaker"]))
val = trv.iloc[va_rel]

clean["split"] = "train"                         # default
clean.loc[val.index,  "split"] = "val"
clean.loc[test.index, "split"] = "test"

# guard: no speaker may appear in more than one split
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    sa = set(clean.loc[clean["split"] == a, "speaker"])
    sb = set(clean.loc[clean["split"] == b, "speaker"])
    assert not (sa & sb), f"speaker leakage between {a}/{b}"
print(clean["split"].value_counts())


In [ ]:
# === Resample each clean clip to 16 kHz mono (<=5 s) -> PROCESSED_DIR (originals untouched) ===
MAX_LEN = int(MAX_SEC * SR)

def resample_file(src_path, emotion):
    out_dir = PROCESSED_DIR / emotion
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / Path(src_path).name
    if out_path.exists():
        return str(out_path)                       # idempotent: skip already-processed
    y, _ = librosa.load(src_path, sr=SR, mono=True)
    y = y[:MAX_LEN]                                 # cap at 5 s, no padding
    sf.write(str(out_path), y, SR)
    return str(out_path)

clean["processed_path"] = [resample_file(p, e) for p, e in
                           tqdm(list(zip(clean["file_path"], clean["emotion"])), desc="resample")]
print("16 kHz audio ->", PROCESSED_DIR.resolve())


In [ ]:
# === Push the CLEANED dataset to local SQLite (after all processing) ===
# Single source of truth for CleanModel.ipynb: one row per clean clip with its 16 kHz path,
# speaker, and 3-way split. SQLite has no inline INDEX clause, so indexes are created separately.
DATASET_TABLE = "audio_dataset_w2v2"
DATASET_DDL = f"""
CREATE TABLE {DATASET_TABLE} (
    clip_id        INTEGER      NOT NULL PRIMARY KEY,
    filename       TEXT,
    file_path      TEXT,
    processed_path TEXT,
    emotion        TEXT,
    source         TEXT,
    duration_sec   REAL,
    speaker        TEXT,
    split          TEXT
)
"""
INDEX_DDLS = [
    f"CREATE INDEX idx_{DATASET_TABLE}_emotion ON {DATASET_TABLE} (emotion)",
    f"CREATE INDEX idx_{DATASET_TABLE}_split   ON {DATASET_TABLE} (split)",
    f"CREATE INDEX idx_{DATASET_TABLE}_speaker ON {DATASET_TABLE} (speaker)",
]
COLS = ["clip_id", "filename", "file_path", "processed_path", "emotion",
        "source", "duration_sec", "speaker", "split"]

recreate_table(DATASET_TABLE, DATASET_DDL, INDEX_DDLS)
clean[COLS].to_sql(DATASET_TABLE, engine, if_exists="append", index=False)
with engine.connect() as conn:
    n = conn.execute(text(f"SELECT COUNT(*) FROM {DATASET_TABLE}")).scalar()
print(f"wrote {n} rows to local SQLite `{DB_PATH.name}` :: `{DATASET_TABLE}`  "
      f"(cleaned + 3-way speaker-disjoint split)")


In [ ]:
# === Reproducibility check (per-split counts + speaker-disjoint guarantee) ===
print("SEED:", SEED)
print("\nclips per split:");    print(clean["split"].value_counts())
print("\nspeakers per split:"); print(clean.groupby("split")["speaker"].nunique())
print("\nemotion x split:");    print(pd.crosstab(clean["emotion"], clean["split"]))


## Reproducibility & sharing

- This notebook **is** the shared selection pipeline for the wav2vec2 / WavLM model — commit it so the team runs the same file:
  ```
  git add Notebooks/SQLCLEAN.ipynb && git commit -m "Local SQL-backed data selection pipeline" && git push
  ```
- `SEED = 42` + `GroupShuffleSplit(test_size=0.15, random_state=42)` (applied twice) → identical, speaker-disjoint splits for everyone who runs this against the same `../Emotions` folder.
- `Notebooks/ser_local.db` is the single source of truth for this pipeline. It's small (metadata only,
  no audio) and reproducible from `../Emotions` in a couple of minutes, so it's **gitignored** rather
  than committed — every teammate builds their own copy by running this notebook once.
- `CleanModel.ipynb` reads straight from this table and does **no re-filtering or re-splitting**:
  ```sql
  SELECT processed_path, emotion, speaker FROM audio_dataset_w2v2 WHERE split = 'train';
  SELECT processed_path, emotion, speaker FROM audio_dataset_w2v2 WHERE split = 'val';
  SELECT processed_path, emotion, speaker FROM audio_dataset_w2v2 WHERE split = 'test';
  ```
- Originals in `Emotions/` are never modified; 16 kHz clips are written to `Emotions_w2v2_16k/`.
